# P1 — EDA y estadística descriptiva

Análisis exploratorio y estadística descriptiva del dataset *Give Me Some
Credit*: calidad de datos, distribución de variables y su relación con la
variable objetivo. Los hallazgos se documentan en `docs/diccionario_datos.md`
y `docs/datasheet.md`, y se resumen en `docs/resumen_ejecutivo.md`.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import SEED, TARGET_COL
from src.data import load_raw_training

np.random.seed(SEED)
sns.set_theme(style='whitegrid')

## 1. Carga de datos

In [2]:
df = load_raw_training()
df.shape

(150000, 11)

## 2. Calidad de datos

- Valores faltantes (`MonthlyIncome`, `NumberOfDependents`).
- Códigos de error 96/98 en variables de días de atraso.
- Outliers en `RevolvingUtilizationOfUnsecuredLines` y `DebtRatio`.

Las decisiones de limpieza derivadas se documentan en
`docs/diccionario_datos.md`.

In [3]:
df.isna().mean().sort_values(ascending=False)

MonthlyIncome                           0.198207
NumberOfDependents                      0.026160
SeriousDlqin2yrs                        0.000000
age                                     0.000000
RevolvingUtilizationOfUnsecuredLines    0.000000
DebtRatio                               0.000000
NumberOfTime30-59DaysPastDueNotWorse    0.000000
NumberOfOpenCreditLinesAndLoans         0.000000
NumberOfTimes90DaysLate                 0.000000
NumberRealEstateLoansOrLines            0.000000
NumberOfTime60-89DaysPastDueNotWorse    0.000000
dtype: float64

In [4]:
late_cols = [
    'NumberOfTime30-59DaysPastDueNotWorse',
    'NumberOfTimes90DaysLate',
    'NumberOfTime60-89DaysPastDueNotWorse',
]
error_code_mask = (df[late_cols] >= 96).any(axis=1)
print('Filas con código de error 96/98:', error_code_mask.sum())
print('Tasa de impago en filas con código de error:', df.loc[error_code_mask, TARGET_COL].mean())
print('Tasa de impago global:', df[TARGET_COL].mean())

Filas con código de error 96/98: 269
Tasa de impago en filas con código de error: 0.5464684014869888
Tasa de impago global: 0.06684


In [5]:
print('age == 0:', (df['age'] == 0).sum())
print('RevolvingUtilizationOfUnsecuredLines > 1:', (df['RevolvingUtilizationOfUnsecuredLines'] > 1).sum(),
      '| max:', df['RevolvingUtilizationOfUnsecuredLines'].max())
print('DebtRatio > 1:', (df['DebtRatio'] > 1).sum(),
      f"({(df['DebtRatio'] > 1).mean():.1%})",
      '| max:', df['DebtRatio'].max())

age == 0: 1
RevolvingUtilizationOfUnsecuredLines > 1: 3321 | max: 50708.0
DebtRatio > 1: 35137 (23.4%) | max: 329664.0


## 3. Estadística descriptiva univariada

In [6]:
df.describe(percentiles=[.01, .05, .5, .95, .99])

,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
count,150000.000000,150000.000000,150000.000000,150000.000000,150000.000000,1.202690e+05,150000.000000,150000.000000,150000.000000,150000.000000,146076.000000
mean,0.066840,6.048438,52.295207,0.421033,353.005076,6.670221e+03,8.452760,0.265973,1.018240,0.240387,0.757222
std,0.249746,249.755371,14.771866,4.192781,2037.818523,1.438467e+04,5.145951,4.169304,1.129771,4.155179,1.115086
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000
1%,0.000000,0.000000,24.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000
5%,0.000000,0.000000,29.000000,0.000000,0.004329,1.300000e+03,2.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.154181,52.000000,0.000000,0.366508,5.400000e+03,8.000000,0.000000,1.000000,0.000000,0.000000
95%,1.000000,1.000000,78.000000,2.000000,2449.000000,1.458760e+04,18.000000,1.000000,3.000000,1.000000,3.000000
99%,1.000000,1.092956,87.000000,4.000000,4979.040000,2.500000e+04,24.000000,3.000000,4.000000,2.000000,4.000000
max,1.000000,50708.000000,109.000000,98.000000,329664.000000,3.008750e+06,58.000000,98.000000,54.000000,98.000000,20.000000


## 4. Estadística descriptiva bivariada (vs. variable objetivo)

In [7]:
df.groupby(TARGET_COL).mean(numeric_only=True)

,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
SeriousDlqin2yrs,,,,,,,,,,
0,6.168855,52.751375,0.280109,357.151168,6747.837774,8.493620,0.135225,1.020368,0.126666,0.743417
1,4.367282,45.926591,2.388490,295.121066,5630.826493,7.882306,2.091362,0.988530,1.828047,0.948208


## 5. Hallazgos descriptivos

1. **Clases muy desbalanceadas.** Solo 6.68% de los clientes presentan impago
   severo `SeriousDlqin2yrs=1`. Cualquier modelo de P3 debe evaluarse con
   métricas robustas al desbalance (AUC-ROC, KS) y no con accuracy simple; y el
   split de P3 debe ser estratificado.
2. **Los códigos de error 96/98 no son ruido aleatorio: concentran riesgo
   extremo.** Las 269 filas con estos códigos en las variables de atraso
   muestran una tasa de impago de 54.6%, ~8 veces la tasa base (6.68%). No son
   conteos válidos (nadie estuvo atrasado "96 veces"), pero tampoco deben
   descartarse sin más: son probablemente clientes con historial de
   incumplimiento severo mal codificado. Tratarlos como faltantes e imputar
   sin perder la señal de riesgo (ej. una variable binaria "tuvo código de
   error") es preferible a simplemente eliminarlos.
3. **`RevolvingUtilizationOfUnsecuredLines` tiene outliers extremos.** 3,321
   registros (2.2%) superan el valor teórico máximo de 1 (utilización de 100%),
   llegando hasta 50,708 — inconsistente con la definición de la variable como
   proporción. Requiere winsorización o tope en el preprocesamiento.
4. **`DebtRatio` está distorsionado por ingresos muy bajos o nulos.** 35,137
   registros (23.4%) superan 1, con un máximo de 329,664. Esto ocurre cuando
   `MonthlyIncome` es cercano a 0, lo que infla artificialmente el ratio —
   confirma que la imputación de `MonthlyIncome` debe hacerse con cuidado antes
   de usar `DebtRatio` como predictor.
5. **El perfil bivariado es consistente con la intuición de negocio:** los
   clientes que caen en impago tienen, en promedio, más atrasos históricos
   (30-59, 60-89 y 90+ días), son más jóvenes (46 vs. 53 años) y tienen menor
   ingreso mensual — variables candidatas fuertes para el modelo predictivo (P3).